# Laboratorium: trasa ciężarówki z Qiskit

Ciężarówka wyjeżdża z **centrum dystrybucyjnego (DC)**, rozwozi towar do salonów i wraca do DC.
W jakiej kolejności odwiedzić salony, żeby przejechać **jak najmniej kilometrów**?
Rozwiążemy to klasycznie, a potem algorytmem kwantowym **QAOA** na symulatorze komputera kwantowego.

| Część | Temat |
|---|---|
| 1 | Dane i rozwiązanie klasyczne |
| 2 | Problem jako zmienne 0/1 – **zadanie 1** |
| 3 | Od modelu do kubitów |
| 4 | Rozwiązanie algorytmem QAOA |
| 5 | Większy problem: 16 salonów – **zadania 2a i 2b** |
| 6 | Dodanie drugiej ciężarówki – **zadanie 3** |

W komórkach **✏️ ZADANIE** piszesz kod sam, od zera. Komórka **✅ Sprawdzenie** pod spodem powie, czy działa.

Przed rozpoczęciem laboratorium skonfiguruj środowisko: **environment.yml**

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from docplex.mp.model import Model
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_optimization.optimizers import COBYLA
from qiskit_optimization.translators import from_docplex_mp

## Część 1. Dane i rozwiązanie klasyczne

DC i trzy salony na uproszczonej mapie (km, odległości w linii prostej).

In [ ]:
PUNKTY = {"DC": (0, 0), "A": (2, 6), "B": (7, 5), "C": (6, 1)}
SALONY = ["A", "B", "C"]
ODL = {(p, q): round(float(np.hypot(PUNKTY[p][0] - PUNKTY[q][0], PUNKTY[p][1] - PUNKTY[q][1])), 1)
       for p in PUNKTY for q in PUNKTY}


def rysuj(trasy=(), tytul=""):
    """Rysuje mapę; trasy to lista tras, np. [["A", "B", "C"]] – każda zaczyna i kończy się w DC."""
    fig, ax = plt.subplots(figsize=(5, 4))
    for nazwa, (px, py) in PUNKTY.items():
        ax.scatter(px, py, s=300 if nazwa == "DC" else 150, c="#142138" if nazwa == "DC" else "#14BBB8", zorder=3)
        ax.annotate(nazwa, (px, py), textcoords="offset points", xytext=(10, 8), fontsize=11)
    for k, trasa in enumerate(trasy):
        pelna = ["DC", *trasa, "DC"]
        ax.plot([PUNKTY[p][0] for p in pelna], [PUNKTY[p][1] for p in pelna],
                color=["#E94A16", "#087D79"][k % 2], lw=2, zorder=2)
    ax.set_title(tytul)
    ax.set_aspect("equal")
    ax.margins(0.15)
    ax.grid(alpha=0.3)
    plt.show()


def dlugosc_trasy(trasa):
    pelna = ["DC", *trasa, "DC"]
    return round(sum(ODL[skad, dokad] for skad, dokad in zip(pelna, pelna[1:])), 1)


# Klasycznie: sprawdzamy wszystkie 3! = 6 kolejności
for trasa in sorted(itertools.permutations(SALONY), key=dlugosc_trasy):
    print(" → ".join(["DC", *trasa, "DC"]), f"{dlugosc_trasy(trasa):5.1f} km")
rysuj([["A", "B", "C"]], "Najkrótsza trasa")

Trasa przejechana w odwrotną stronę ma tę samą długość, więc najlepsze są **dwie**.
Dla 10 salonów tras jest już 10! = 3,6 mln, a dla 20 – ponad 10^18.

## Część 2. Problem jako zmienne 0/1

Komputer kwantowy rozumie tylko **zmienne 0/1**: `x[i, t] = 1`, jeśli salon `i` odwiedzamy w kroku `t`.
3 salony × 3 kroki = **9 zmiennych**. Trasa DC → A → B → C → DC:

| | krok 0 | krok 1 | krok 2 |
|---|---|---|---|
| **A** | 1 | 0 | 0 |
| **B** | 0 | 1 | 0 |
| **C** | 0 | 0 | 1 |

Model zapisujemy w bibliotece **docplex**. Funkcja celu jest gotowa: iloczyn `x[i, t] * x[j, t + 1]` jest równy 1
tylko wtedy, gdy z salonu `i` jedziemy prosto do `j` – wtedy doliczamy `ODL[i, j]`.

In [ ]:
KROKI = range(len(SALONY))

model = Model("trasa")
x = model.binary_var_dict([(i, t) for i in SALONY for t in KROKI], name="x")
model.minimize(
    model.sum(ODL["DC", i] * x[i, 0] for i in SALONY)                      # DC → pierwszy salon
    + model.sum(ODL[i, j] * x[i, t] * x[j, t + 1]                           # przejazdy między salonami
                for i in SALONY for j in SALONY if i != j for t in range(len(SALONY) - 1))
    + model.sum(ODL[i, "DC"] * x[i, len(SALONY) - 1] for i in SALONY))     # ostatni salon → DC

### ✏️ Zadanie 1 – ograniczenia

Bez ograniczeń najkrótsza „trasa” to same zera – nigdzie nie jedziemy. Dodaj do modelu dwa rodzaje ograniczeń:
1. każdy salon odwiedzamy **dokładnie raz** – dla każdego salonu `i` suma `x[i, t]` po krokach jest równa 1,
2. w każdym kroku jesteśmy **w dokładnie jednym** salonie – dla każdego kroku `t` suma `x[i, t]` po salonach jest równa 1.

Przydatne: `model.sum(wyrażenie for ... in ...)` oraz `model.add_constraint(wyrażenie == 1)`.

In [ ]:
# ✏️ ZADANIE
# Twój kod:


In [ ]:
# ✅ Sprawdzenie
assert model.number_of_constraints == 6, f"Model ma {model.number_of_constraints} ograniczeń, a powinien mieć 6"
problem = from_docplex_mp(model)
najkrotsza = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(problem)  # dokładne rozwiązanie Twojego modelu
wzorzec = min(dlugosc_trasy(p) for p in itertools.permutations(SALONY))       # najkrótsza trasa z części 1
assert abs(najkrotsza.fval - wzorzec) < 0.05, \
    f"Najlepsza „trasa” w Twoim modelu ma {najkrotsza.fval:.1f} km zamiast {wzorzec} km – sprawdź ograniczenia"
print("✅ Dobrze!\n")
print(problem.prettyprint())

## Część 3. Od modelu do kubitów

1. **QUBO** – algorytmy kwantowe nie znają ograniczeń, więc zamieniamy je na **kary**: za trasę łamiącą ograniczenie
   doliczamy dużą liczbę. Najlepsze rozwiązanie QUBO jest wtedy zawsze poprawną trasą.
2. **Model Isinga** – ten sam problem w języku fizyki: każda zmienna 0/1 to **jeden kubit**, a funkcja celu to
   „energia” kubitów. Najkrótsza trasa = stan o **najniższej energii**.

Symulator kwantowy przechowuje 2^kubity liczb po 16 bajtów: 9 kubitów to 8 KB, a 30 kubitów to już 16 GB.

In [ ]:
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
hamiltonian, _ = qubo.to_ising()

print("Zmienne w QUBO:", qubo.get_num_vars(), "| ograniczenia:", qubo.get_num_linear_constraints())
print("Kara dobrana automatycznie:", converter.penalty)
print("Liczba kubitów:", hamiltonian.num_qubits)

## Część 4. Rozwiązanie algorytmem QAOA

QAOA to algorytm **hybrydowy**:
1. buduje z Hamiltonianu **obwód kwantowy** z kilkoma pokrętłami (parametrami),
2. uruchamia go wiele razy – każdy pomiar kubitów daje jedną trasę (ciąg 0/1),
3. klasyczny optymalizator (COBYLA) kręci pokrętłami tak, żeby obwód częściej dawał krótkie trasy.

`reps` to liczba warstw obwodu. `MinimumEigenOptimizer` sam wykonuje konwersję do QUBO i Isinga z części 3.

In [ ]:
qaoa = QAOA(sampler=StatevectorSampler(seed=7), optimizer=COBYLA(maxiter=200), reps=1)
wynik = MinimumEigenOptimizer(qaoa).solve(problem)

# x[i, t] = 1 → salon i w kroku t; sortujemy wybrane salony po krokach
trasa = [i for (i, t), v in sorted(zip(x.keys(), wynik.x), key=lambda k: k[0][1]) if v > 0.5]
print("QAOA:", " → ".join(["DC", *trasa, "DC"]), f"({wynik.fval:.1f} km)")
rysuj([trasa], f"QAOA: {wynik.fval:.1f} km")

## Część 5. Większy problem: 16 salonów

**Planowanie dni dostaw:** 16 salonów leży wzdłuż jednej trasy. Każdy dostaje towar **w poniedziałek (0) albo
w czwartek (1)** – czyli **1 salon = 1 kubit**.
- każdy salon ma określony **koszt dostawy w czwartek, względem dostawy w poniedziałek** (liczba ujemna = tańsza dostawa w czwartek),
- jeśli **sąsiednie salony** mają różne dni, ciężarówka jedzie odcinek między nimi dwa razy – płacimy **odległość**.

Każdy ciąg 0/1 jest poprawnym planem, więc nie ma ograniczeń. Model jest gotowy.

In [ ]:
N = 16
rng = np.random.default_rng(5)
KOSZT_CZWARTKU = np.round(rng.uniform(-3, 3, N), 1)  # Koszt czwartku względem poniedziałku
ODL_SASIEDZI = np.round(rng.uniform(0.5, 2, N - 1), 1)  # km między salonem i a i + 1

model_dni = Model("dni_dostaw")
z = model_dni.binary_var_list(N, name="z")  # z[i] = 1 → dostawa w czwartek
model_dni.minimize(
    model_dni.sum(KOSZT_CZWARTKU[i] * z[i] for i in range(N))
    # z[i] + z[i+1] - 2·z[i]·z[i+1] = 1 dokładnie wtedy, gdy sąsiedzi mają różne dni
    + model_dni.sum(ODL_SASIEDZI[i] * (z[i] + z[i + 1] - 2 * z[i] * z[i + 1]) for i in range(N - 1)))
problem_dni = from_docplex_mp(model_dni)
print("Zmiennych (kubitów):", problem_dni.get_num_vars())

### ✏️ Zadanie 2a – rozwiąż plan dostaw QAOA

Na wzór części 4 rozwiąż `problem_dni` algorytmem QAOA z ustawieniami:
`reps=2`, `initial_point=[0.1, 0.1, 0.4, 0.4]` (punkt startowy pokręteł), `StatevectorSampler(seed=7)`,
`COBYLA(maxiter=200)`.
Wynik zapisz w zmiennej `wynik_dni` i wypisz numery salonów z dostawą w czwartek (`wynik_dni.x[i] == 1`).

In [ ]:
# ✏️ ZADANIE
# Twój kod:


In [ ]:
# ✅ Sprawdzenie + wykres: słupek = koszt czwartku dla salonu, kolor = wybrany dzień
assert len(wynik_dni.x) == N, "wynik_dni powinien zawierać decyzję dla każdego z 16 salonów"
print(f"✅ Dobrze! Koszt planu: {wynik_dni.fval:.1f}")
plan = wynik_dni.x.astype(int)
fig, ax = plt.subplots(figsize=(9, 3.5))
for dzien, kolor, nazwa in ((0, "#142138", "poniedziałek"), (1, "#14BBB8", "czwartek")):
    ax.bar(np.flatnonzero(plan == dzien), KOSZT_CZWARTKU[plan == dzien], color=kolor, label=nazwa)
ax.axhline(0, color="#5D6774", lw=0.8)
ax.set_xticks(range(N))
ax.set_xlabel("salon (kolejność wzdłuż trasy)")
ax.set_ylabel("koszt dostawy w czwartek")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
plt.show()

### ✏️ Zadanie 2b – wpływ parametrów

Pozmieniaj definicje parametrów KOSZT_CZWARTKU i ODL_SASIEDZI i zobacz jak będą zmieniać się wyniki.

## Część 6. Dodanie drugiej ciężarówki

### ✏️ Zadanie 3 – dwie ciężarówki

Dochodzi salon **D**, a towar wiozą **dwie ciężarówki**; każda obsługuje **dokładnie 2 salony** i wraca do DC.
Zmienne: `y[k, i, t] = 1`, jeśli ciężarówka `k` odwiedza salon `i` w swoim kroku `t` – razem 2 × 4 × 2 = 16 kubitów.

Funkcja celu jest gotowa. Dodaj ograniczenia:
1. każdy salon odwiedza dokładnie jedna ciężarówka, dokładnie raz (suma po `k` i `t` równa 1),
2. każda ciężarówka w każdym kroku jest w dokładnie jednym salonie (suma po `i` równa 1).

In [ ]:
PUNKTY["D"] = (-4, 3)
SALONY_VRP, CIEZAROWKI, KROKI_VRP = ["A", "B", "C", "D"], [1, 2], [0, 1]
ODL = {(p, q): round(float(np.hypot(PUNKTY[p][0] - PUNKTY[q][0], PUNKTY[p][1] - PUNKTY[q][1])), 1)
       for p in PUNKTY for q in PUNKTY}

model_vrp = Model("dwie_ciezarowki")
y = model_vrp.binary_var_dict([(k, i, t) for k in CIEZAROWKI for i in SALONY_VRP for t in KROKI_VRP], name="y")
model_vrp.minimize(
    model_vrp.sum(ODL["DC", i] * y[k, i, 0] + ODL[i, "DC"] * y[k, i, 1] for k in CIEZAROWKI for i in SALONY_VRP)
    + model_vrp.sum(ODL[i, j] * y[k, i, 0] * y[k, j, 1]
                    for k in CIEZAROWKI for i in SALONY_VRP for j in SALONY_VRP if i != j))

In [ ]:
# ✏️ ZADANIE
# Twój kod:


In [ ]:
# ✅ Sprawdzenie + rozwiązanie dokładne (klasyczne przeszukanie 2^16 stanów)
assert model_vrp.number_of_constraints == 8, f"Model ma {model_vrp.number_of_constraints} ograniczeń, a powinien mieć 8"
wynik_vrp = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(from_docplex_mp(model_vrp))
wzorzec_vrp = min(dlugosc_trasy(p[:2]) + dlugosc_trasy(p[2:]) for p in itertools.permutations(SALONY_VRP))
assert abs(wynik_vrp.fval - wzorzec_vrp) < 0.05, \
    f"Najlepsze rozwiązanie Twojego modelu ma {wynik_vrp.fval:.1f} km zamiast {wzorzec_vrp:.1f} km – sprawdź ograniczenia"
trasy_vrp = [[i for (kk, i, t), v in sorted(zip(y.keys(), wynik_vrp.x), key=lambda e: e[0][2]) if kk == k and v > 0.5]
             for k in CIEZAROWKI]
print("✅ Dobrze!", *[f"Ciężarówka {k}: " + " → ".join(["DC", *tr, "DC"]) for k, tr in zip(CIEZAROWKI, trasy_vrp)], sep="\n")
rysuj(trasy_vrp, f"Dwie ciężarówki: {wynik_vrp.fval:.1f} km")